# Numérique-01 C# — Équation de Burgers : volumes finis, solveur de Riemann, et ce qu'une preuve prouve


**Navigation** : [Index de la série](README.md) | [Version Python](Numerique-01-Burgers-Volumes-Finis-Python.ipynb)
> **Twin C#** de [`Numérique-01 — Burgers, volumes finis (Python)`](Numerique-01-Burgers-Volumes-Finis-Python.ipynb) (numpy + matplotlib).
> BCL .NET seule, **0 NuGet**, **0 réseau** : `double[]`, `System.Linq`, `Math.*`.
> Les figures du jumeau Python sont remplacées par des **sorties texte** ; les mêmes nombres sont rejoués
> et chacun est porté par une **assertion exécutable**. Le carnet est déterministe : chaque exécution rejoue les mêmes valeurs.

## Objectifs pédagogiques

1. Dériver la condition de Rankine-Hugoniot et la solution exacte du problème de Riemann pour l'équation de Burgers.
2. Implémenter les schémas de Godunov (avec solveur de Riemann) et de Rusanov, et la condition CFL.
3. Mesurer le taux de convergence L1 des deux schémas sur choc et sur raréfaction.
4. Mesurer le temps de formation d'un choc par la méthode des caractéristiques, et le comparer à la prédiction analytique.
5. Énoncer, lacune par lacune, la frontière entre ce qu'une preuve formelle du modèle continu établit et ce qu'un schéma numérique mesure.

## Prérequis

Dérivées partielles, intégration, C# de niveau standard du dépôt (tableaux `double[]`, LINQ).

## Durée estimée : 45-60 minutes

Ce carnet est le jumeau C# de l'ouvreur de la sous-série *Numérique* de la série Physique. Il est volontairement CPU : aucun matériel dédié, aucune stochasticité — chaque exécution rejoue les mêmes nombres.


### Environnement opérationnel

Le carnet n'utilise que la BCL .NET (aucun NuGet, aucun réseau). La cellule suivante imprime la version effective :
toute exécution doit montrer les mêmes valeurs calculées sur ce dépôt (pas d'aléa, schémas déterministes).


In [1]:
using System;
using System.Globalization;
using System.Linq;

// Formatage invariant (règle marathon : la culture FR ne persiste pas entre cellules .NET Interactive)
static string FI(double x, string fmt = "F6") => x.ToString(fmt, CultureInfo.InvariantCulture);

// Garde exécutable : lève si la condition échoue (équivalent des assert Python du jumeau)
static void Check(bool cond, string msg) { if (!cond) throw new InvalidOperationException(msg); }

Console.WriteLine($".NET  : {Environment.Version}");
Console.WriteLine("BCL   : System.*, System.Linq (0 NuGet, 0 reseau)");
Console.WriteLine("Imports reussis : BCL seule est presente (disponibilite seule : ce n'est pas un test de bon fonctionnement).");


The below script needs to be able to find the current output cell; this is an easy method to get it.

.NET  : 10.0.15


BCL   : System.*, System.Linq (0 NuGet, 0 reseau)


Imports reussis : BCL seule est presente (disponibilite seule : ce n'est pas un test de bon fonctionnement).


## Partie 1 — Le problème continu : conservation et caractéristiques

L'équation de Burgers visqueuse s'écrit $u_t + u\,u_x = \varepsilon\,u_{xx}$.
Sa limite $\varepsilon \to 0$ est la **loi de conservation scalaire**

$$u_t + f(u)_x = 0, \qquad f(u) = \tfrac{u^2}{2},$$

c'est-à-dire $u_t + u\,u_x = 0$. Trois lectures de la même équation :

- **forme différentielle** : $u_t + u\,u_x = 0$ ;
- **forme conservative** : $u_t + \left(\tfrac{u^2}{2}\right)_x = 0$ ;
- **forme intégrale** sur tout segment $[a,b]$ : $\dfrac{d}{dt}\int_a^b u\,dx = f(u(a)) - f(u(b))$.

La forme intégrale est la seule qui survit à l'apparition de discontinuités : c'est elle qui définit la solution quand les dérivées n'existent plus.

### 1.1 Caractéristiques

Le long des courbes $x(t) = x_0 + u_0(x_0)\,t$, la solution reste constante ($u = u_0(x_0)$).
Deux conséquences, visibles dans la sortie suivante :

- si $u_0$ décroît assez vite quelque part, deux caractéristiques finissent par se **croiser** : la solution classique meurt, un **choc** naît ;
- si $u_0$ croît, les caractéristiques s'écartent : une **raréfaction** comble le vide.

In [2]:
// Données initiales de Riemann (mêmes entrées que le jumeau Python : 17 particules de linspace(-1, 1, 17))
static double U0Choc(double x)         => x < 0.0 ? 1.0 : 0.0;   // uL = 1 à gauche
static double U0Rarefaction(double x)  => x < 0.0 ? 0.0 : 1.0;   // uR = 1 à droite

var x0s = Enumerable.Range(0, 17).Select(k => -1.0 + 2.0 * k / 16.0).ToArray();  // linspace(-1, 1, 17)

foreach (var (nom, u0) in new[] { ("choc : uL=1 > uR=0", (Func<double, double>)U0Choc),
                                   ("raréfaction : uL=0 < uR=1", U0Rarefaction) })
{
    double t = 1.0;
    var feet = x0s.Select(x0 => x0 + u0(x0) * t).ToArray();      // position de la particule partie de x0
    double minGap = double.PositiveInfinity; int iMin = -1;
    for (int i = 1; i < feet.Length; i++)
    {
        double gap = feet[i] - feet[i - 1];
        if (gap < minGap) { minGap = gap; iMin = i; }
    }
    Console.WriteLine($"{nom,-24} : à t=1, pied min x={FI(feet.Min(), "F3")}, pied max x={FI(feet.Max(), "F3")}, "
                      + $"plus petit écart entre voisins = {FI(minGap, "F3")} (entre x0={FI(x0s[iMin - 1], "F2")} et x0={FI(x0s[iMin], "F2")})");
}

// Garde exécutable : le cas choc croise (un écart négatif), le cas raréfaction jamais (tous > 0)
var feetChoc = x0s.Select(x0 => x0 + U0Choc(x0) * 1.0).ToArray();
var feetFan  = x0s.Select(x0 => x0 + U0Rarefaction(x0) * 1.0).ToArray();
double gapChoc = Enumerable.Range(1, 16).Select(i => feetChoc[i] - feetChoc[i - 1]).Min();
double gapFan  = Enumerable.Range(1, 16).Select(i => feetFan[i] - feetFan[i - 1]).Min();
Check(gapChoc < 0.0, "choc : un écart négatif entre particules voisines est attendu");
Check(gapFan > 0.0, "raréfaction : aucun écart négatif n'est attendu");
Console.WriteLine($"choc         : la famille gauche (pente 1) rattrape la famille immobile dès x = 0 — écart minimal {FI(gapChoc, "F3")} < 0 : croisement.");
Console.WriteLine($"raréfaction  : la famille immobile est dépassée par la famille de pente 1 — écart minimal {FI(gapFan, "F3")} > 0 : aucun croisement (à l'intérieur d'une famille l'écart reste le pas de discrétisation).");


choc : uL=1 > uR=0       : à t=1, pied min x=0.000, pied max x=1.000, plus petit écart entre voisins = -0.875 (entre x0=-0.12 et x0=0.00)


raréfaction : uL=0 < uR=1 : à t=1, pied min x=-1.000, pied max x=2.000, plus petit écart entre voisins = 0.125 (entre x0=-1.00 et x0=-0.88)


choc         : la famille gauche (pente 1) rattrape la famille immobile dès x = 0 — écart minimal -0.875 < 0 : croisement.


raréfaction  : la famille immobile est dépassée par la famille de pente 1 — écart minimal 0.125 > 0 : aucun croisement (à l'intérieur d'une famille l'écart reste le pas de discrétisation).


**Lecture.** Dans le cas choc, la famille gauche — plus rapide — rattrape la famille immobile : l'information se surcharge et un choc se forme immédiatement en $x=0$ (écart minimal négatif). Dans le cas raréfaction, aucune particule n'en rattrape une autre (tous les écarts restent positifs) : les deux états sont reliés par un éventail.

### 1.2 Condition de Rankine-Hugoniot — dérivation

Intégrons la forme conservative sur le petit segment mobile $[x_s(t)-\varepsilon,\,x_s(t)+\varepsilon]$
qui enjambe une discontinuité se déplaçant à la vitesse $s = dx_s/dt$ :

$$\frac{d}{dt}\int_{x_s-\varepsilon}^{x_s+\varepsilon} u\,dx = f\big(u(x_s-\varepsilon)\big) - f\big(u(x_s+\varepsilon)\big).$$

Quand $\varepsilon \to 0$, le membre de gauche tend vers $s\,(u_L - u_R)$ (dérivation d'une intégrale à bornes mobiles), d'où

$$\boxed{\;s\,(u_L - u_R) = f(u_L) - f(u_R)\;}$$

Pour Burgers, $f(u)=u^2/2$ et donc $s = \dfrac{u_L + u_R}{2}$ : la vitesse du choc est la **moyenne arithmétique** des deux états.

In [3]:
// Flux de Burgers f(u) = u^2 / 2
static double BurgersFlux(double u) => 0.5 * u * u;

// Solution exacte du problème de Riemann u_t + (u^2/2)_x = 0 en (x, t) :
// ul > ur : choc à la vitesse s = (ul + ur) / 2 ; ul < ur : éventail, u = x/t entre les deux bords.
static double BurgersRiemannExact(double ul, double ur, double x, double t)
{
    if (t <= 0.0) return x < 0.0 ? ul : ur;
    if (ul > ur)
    {
        double s = 0.5 * (ul + ur);
        return x < s * t ? ul : ur;
    }
    // raréfaction : éventail entre ul*t et ur*t (ul < ur)
    double xi = x / t;
    if (xi <= ul) return ul;
    if (xi >= ur) return ur;
    return xi;
}

// validations fermées, imprimées : ces nombres servent de garde-fou
double sChoc = 0.5 * (1.0 + 0.0);
Console.WriteLine($"vitesse du choc (uL=1, uR=0) : s = {FI(sChoc)}");
double uFan = BurgersRiemannExact(0.0, 1.0, 0.0, 1.0);
Console.WriteLine($"ventre de la raréfaction (uL=0, uR=1) en x=0, t=1 : u = {FI(uFan)}");

Check(Math.Abs(sChoc - 0.5) < 1e-15, "s_choc doit valoir exactement 0.5");
Check(Math.Abs(uFan - 0.0) < 1e-15, "u_fan doit valoir exactement 0");
Console.WriteLine("Formules 1.2 évaluées sur deux points fermés : cohérence interne seule, aucun schéma ni simulation n'entre dans ces garde-fous. La confrontation à un choc numérique a lieu en Partie 2.");


vitesse du choc (uL=1, uR=0) : s = 0.500000


ventre de la raréfaction (uL=0, uR=1) en x=0, t=1 : u = 0.000000


Formules 1.2 évaluées sur deux points fermés : cohérence interne seule, aucun schéma ni simulation n'entre dans ces garde-fous. La confrontation à un choc numérique a lieu en Partie 2.


**Lecture.** La sortie confirme les deux pierres angulaires utilisées ensuite :
pour un choc, la position avance à $s\,t$ avec $s=(u_L+u_R)/2$ ; dans un éventail, la valeur au point $x$ à l'instant $t$ est $x/t$.

**Sélection d'entropie.** Le problème de Riemann admet mathématiquement plusieurs solutions faibles.
La physique (limite $\varepsilon \to 0$ de Burgers visqueuse) sélectionne :

- $u_L > u_R$ : choc, à la vitesse de Rankine-Hugoniot ;
- $u_L < u_R$ : raréfaction.

C'est la **solution entropique** — celle que nos schémas devront approcher.

## Partie 2 — Volumes finis : Godunov et Rusanov

### 2.1 Le principe

On découpe l'axe en cellules $[x_{i-1/2}, x_{i+1/2}]$ et on porte une **moyenne** $U_i^n$ par cellule.
La forme intégrale donne exactement

$$U_i^{n+1} = U_i^n - \frac{\Delta t}{\Delta x}\big(F_{i+1/2} - F_{i-1/2}\big),$$

où $F_{i+1/2}$ est une approximation du **flux moyen** à l'interface. Tout le jeu est dans ce flux :

- **Godunov** : résoudre le problème de Riemann à l'interface $(U_i^n, U_{i+1}^n)$ et évaluer le flux exact sur l'état $u^*$ qui se trouve à $x/t = 0$ ;
- **Rusanov** (Friedrichs local) : $F = \tfrac{1}{2}\big(f(u_L)+f(u_R)\big) - \tfrac{1}{2}a\,(u_R-u_L)$ avec $a = \max(|u_L|,|u_R|)$, la vitesse de propagation maximale locale.

### 2.2 Condition CFL

L'information ne doit pas traverser plus d'une cellule par pas de temps :

$$\Delta t \le C_{\text{CFL}}\,\frac{\Delta x}{\max_i |U_i^n|}, \qquad C_{\text{CFL}} \le 1.$$

Toutes les expériences ci-dessous prennent $C_{\text{CFL}} = 0{,}9$.

In [4]:
// État u* de la solution de Riemann de Burgers évalué en x/t = 0 (schéma de Godunov)
static double BurgersStarInterface(double ul, double ur)
{
    if (ul > ur)                                    // choc
    {
        double s = 0.5 * (ul + ur);
        return s > 0.0 ? ul : ur;
    }
    // éventail
    if (ul >= 0.0) return ul;
    if (ur <= 0.0) return ur;
    return 0.0;
}

static double FluxGodunov(double ul, double ur) => BurgersFlux(BurgersStarInterface(ul, ur));

static double FluxRusanov(double ul, double ur)
{
    double a = Math.Max(Math.Abs(ul), Math.Abs(ur));
    return 0.5 * (BurgersFlux(ul) + BurgersFlux(ur)) - 0.5 * a * (ur - ul);
}

// Interpolation linéaire aux bords plats (sémantique de numpy.interp, même formule)
static double Interp1(double x, double[] xc, double[] u)
{
    int n = xc.Length;
    if (x <= xc[0]) return u[0];
    if (x >= xc[n - 1]) return u[n - 1];
    int lo = 0, hi = n - 1;
    while (hi - lo > 1)
    {
        int mid = (lo + hi) / 2;
        if (xc[mid] <= x) lo = mid; else hi = mid;
    }
    return u[lo] + (u[lo + 1] - u[lo]) * (x - xc[lo]) / (xc[lo + 1] - xc[lo]);
}

// Un pas de temps volume finis d'ordre 1 ; retourne (u_suivant, dt_utilisé).
// dt_max borne le pas : c'est ce qui permet de finir EXACTEMENT à t_final, au lieu
// de le dépasser (même motif que SimulePeriodique, Partie 3).
static (double[] uNew, double dt) AvanceVolumeFinis(double[] u, double dx, double cfl,
                                                    Func<double, double, double> fluxInterface, double? dtMax = null)
{
    double vmax = u.Max(Math.Abs);
    double dt = cfl * dx / Math.Max(vmax, 1e-14);
    if (dtMax.HasValue) dt = Math.Min(dt, dtMax.Value);
    int n = u.Length;
    double[] F = new double[n - 1];                       // flux aux interfaces intérieures
    for (int k = 0; k < n - 1; k++) F[k] = fluxInterface(u[k], u[k + 1]);
    double[] uNew = (double[])u.Clone();
    for (int j = 1; j < n - 1; j++) uNew[j] = u[j] - (dt / dx) * (F[j] - F[j - 1]);
    uNew[0] = u[0];                                       // bords gelés (Neumann)
    uNew[n - 1] = u[n - 1];
    return (uNew, dt);
}

// Boucle jusqu'à t_final ; retourne (u_final, nb_pas)
static (double[] u, int nb) Simule(double[] u0e, double tFinal, double cfl,
                                   Func<double, double, double> fluxInterface, double dx)
{
    double[] u = (double[])u0e.Clone();
    double t = 0.0; int nb = 0;
    while (t < tFinal - 1e-15)
    {
        var (uNew, dt) = AvanceVolumeFinis(u, dx, cfl, fluxInterface, dtMax: tFinal - t);
        u = uNew; t += dt; nb++;
    }
    return (u, nb);
}

// --- expérience : problème de Riemann (uL=1.2, uR=0.4), jusqu'à T = 0.5 ---
double UL = 1.2, UR = 0.4, TFinal = 0.5;
double XMin = -1.0, XMax = 2.0;
int N = 400;
double DX = (XMax - XMin) / N;
var xC = Enumerable.Range(0, N).Select(i => XMin + (i + 0.5) * DX).ToArray();
// Grille fine : répliquer l'arithmétique de np.linspace (start + i*step), pas (stop-start)*i/(n-1).
// Le point i=560 tombe à UN ULP du choc x = s*t = 0.4 : les deux formules y donnent des valeurs
// de part et d'autre de la discontinuité, et l'erreur L1 saute de 0.0006 — l'égalité au jumeau
// Python exige la même suite de doubles.
double stepF = (XMax - XMin) / 1200.0;
var xF = Enumerable.Range(0, 1201).Select(i => XMin + i * stepF).ToArray();
var u0e = xC.Select(x => x < 0.0 ? UL : UR).ToArray();

var (uG, nbG) = Simule(u0e, TFinal, 0.9, FluxGodunov, DX);
var (uR, nbR) = Simule(u0e, TFinal, 0.9, FluxRusanov, DX);
double dxF = xF[1] - xF[0];
double errG = xF.Select(x => Math.Abs(Interp1(x, xC, uG) - BurgersRiemannExact(UL, UR, x, TFinal))).Sum() * dxF;
double errR = xF.Select(x => Math.Abs(Interp1(x, xC, uR) - BurgersRiemannExact(UL, UR, x, TFinal))).Sum() * dxF;
Console.WriteLine($"Godunov : {nbG} pas, erreur L1 = {FI(errG)}");
Console.WriteLine($"Rusanov : {nbR} pas, erreur L1 = {FI(errR)}");

// Garde exécutable : mêmes valeurs que le jumeau Python (deterministe, egalite exigible)
Check(nbG == 89 && nbR == 89, $"pas attendus 89/89, obtenus {nbG}/{nbR}");
Check(Math.Abs(errG / 0.004156312202304002 - 1.0) < 1e-9, $"erreur L1 Godunov divergente du jumeau Python : {errG}");
Check(Math.Abs(errR / 0.00513084770222301 - 1.0) < 1e-9, $"erreur L1 Rusanov divergente du jumeau Python : {errR}");
Check(errG < errR, "Godunov doit étaler moins que Rusanov sur ce choc");


Godunov : 89 pas, erreur L1 = 0.004156


Rusanov : 89 pas, erreur L1 = 0.005131


**Lecture.** Les deux schémas suivent le choc — la position prédite $s\,t$ par la dérivation 1.2 tombe au milieu du saut —
mais ils l'étalent sur quelques cellules : c'est la **dissipation numérique**. Godunov, qui exploite le solveur de Riemann exact,
étale un peu moins que Rusanov : l'écart est quantifié par les erreurs L1 imprimées ci-dessus.

### 2.3 Mesure du taux de convergence

Pour un schéma d'ordre 1, la théorie annonce une erreur $L^1 \propto \Delta x$ sur les cas réguliers,
et sur un choc pur l'erreur reste $\propto \Delta x$ (largeur du choc étalée sur un nombre ~constant de cellules).
La cellule suivante **mesure** ce taux sur deux cas de Riemann (choc, raréfaction) par régression log-log.

In [5]:
// Erreur L1 vs solution exacte pour des N croissants ; taux par régression log-log.
static (double[] pas, double[] errs, double pente) EtudeConvergence(
    Func<double, double, double> fluxInterface, double ul, double ur,
    double tFinal = 0.5, int[] grids = null, double xMin = -1.0, double xMax = 2.0)
{
    grids ??= new[] { 100, 200, 400, 800, 1600 };
    double stepRef = (xMax - xMin) / 2000.0;   // arithmétique np.linspace
    var xRef = Enumerable.Range(0, 2001).Select(i => xMin + i * stepRef).ToArray();
    double dxRef = xRef[1] - xRef[0];
    var pas = new List<double>(); var errs = new List<double>(); var tempsAtteints = new List<double>();
    foreach (int n in grids)
    {
        double dx = (xMax - xMin) / n;
        var xc = Enumerable.Range(0, n).Select(i => xMin + (i + 0.5) * dx).ToArray();
        var u = xc.Select(x => x < 0.0 ? ul : ur).ToArray();
        double t = 0.0;
        while (t < tFinal - 1e-15)
        {
            var (uNew, dt) = AvanceVolumeFinis(u, dx, 0.9, fluxInterface, dtMax: tFinal - t);
            u = uNew; t += dt;
        }
        double e = xRef.Select(x => Math.Abs(Interp1(x, xc, u) - BurgersRiemannExact(ul, ur, x, tFinal))).Sum() * dxRef;
        errs.Add(e); pas.Add(dx); tempsAtteints.Add(t);
    }
    // pente log-log par moindres carrés (fermée)
    double pente = PenteLogLog(pas.ToArray(), errs.ToArray());
    double ecartMax = tempsAtteints.Max(tt => Math.Abs(tt - tFinal));
    Console.WriteLine($"    controle temporel : max |t_atteint - t_final| = {ecartMax:0.00e+00}");
    return (pas.ToArray(), errs.ToArray(), pente);
}

static double PenteLogLog(double[] xs, double[] ys)
{
    double mx = xs.Select(x => Math.Log(x)).Average();
    double my = ys.Select(y => Math.Log(y)).Average();
    double num = 0.0, den = 0.0;
    for (int i = 0; i < xs.Length; i++)
    {
        num += (Math.Log(xs[i]) - mx) * (Math.Log(ys[i]) - my);
        den += (Math.Log(xs[i]) - mx) * (Math.Log(xs[i]) - mx);
    }
    return num / den;
}

var resultats = new List<(string cas, string nom, double pente)>();
foreach (var (cas, ul, ur) in new[] { ("choc", 1.2, 0.4), ("raréfaction", 0.4, 1.2) })
    foreach (var (nom, flux) in new[] { ("Godunov", (Func<double, double, double>)FluxGodunov), ("Rusanov", FluxRusanov) })
    {
        var (_, _, p) = EtudeConvergence(flux, ul, ur);
        resultats.Add((cas, nom, p));
    }

Console.WriteLine("Taux mesurés (régression log-log, N=100..1600) :");
foreach (var (cas, nom, p) in resultats)
    Console.WriteLine($"  {nom,-8} sur {cas,-11} : ordre {FI(p, "F2")}");

// Garde exécutable : mêmes ordres que le jumeau Python (tolérance = l'arrondi d'affichage, 5e-3)
var refPentes = new[] { ("choc", "Godunov", 0.9893506405838333), ("choc", "Rusanov", 0.9944686900004998),
                        ("raréfaction", "Godunov", 0.7514846894597305), ("raréfaction", "Rusanov", 0.7896713904403622) };
foreach (var (cas, nom, pRef) in refPentes)
{
    double p = resultats.First(r => r.cas == cas && r.nom == nom).pente;
    Check(Math.Abs(p - pRef) < 5e-3, $"pente {nom}/{cas} divergente du jumeau Python : {p} vs {pRef}");
}

// --- découpe régionale : l'écart de pente sur la raréfaction est-il localisé ? ---
// L'explication "ordre 1 dans le fan, dégradation aux coins" se TESTE plutôt qu'elle ne s'affirme :
// on redécoupe la même erreur L1 par région, sur la référence exacte et les mêmes grilles.
Console.WriteLine();
Console.WriteLine("Découpe régionale de l'erreur L1 (Godunov, rarefaction 0.4/1.2, memes grilles) :");
var regions = new[] { ("interieur du fan", 0.26, 0.54), ("coin gauche", 0.10, 0.30), ("coin droit", 0.50, 0.70) };
double stepReg = (XMax - XMin) / 2000.0;   // arithmétique np.linspace
var xReg = Enumerable.Range(0, 2001).Select(i => XMin + i * stepReg).ToArray();
double dxReg = xReg[1] - xReg[0];
int[] grillesReg = { 100, 200, 400, 800, 1600 };
var pentesRegion = new List<(string nom, double pente)>();
foreach (var (nomReg, a, b) in regions)
{
    var errsReg = new List<double>();
    foreach (int n in grillesReg)
    {
        double dx = (XMax - XMin) / n;
        var xc = Enumerable.Range(0, n).Select(i => XMin + (i + 0.5) * dx).ToArray();
        var u = xc.Select(x => x < 0.0 ? 0.4 : 1.2).ToArray();
        double t = 0.0;
        while (t < 0.5 - 1e-15)
        {
            var (uNew, dt) = AvanceVolumeFinis(u, dx, 0.9, FluxGodunov, dtMax: 0.5 - t);
            u = uNew; t += dt;
        }
        errsReg.Add(xReg.Where(x => x >= a && x <= b)
                        .Select(x => Math.Abs(Interp1(x, xc, u) - BurgersRiemannExact(0.4, 1.2, x, 0.5))).Sum() * dxReg);
    }
    double penteReg = PenteLogLog(grillesReg.Select(n => (XMax - XMin) / n).ToArray(), errsReg.ToArray());
    pentesRegion.Add((nomReg, penteReg));
}
foreach (var (nomReg, penteReg) in pentesRegion)
    Console.WriteLine($"  {nomReg,-18} : ordre {FI(penteReg, "F2")}");

// Garde exécutable : ordres régionaux du jumeau Python
foreach (var (nomReg, pRef) in new[] { ("interieur du fan", 0.751307015767294), ("coin gauche", 0.7505078016103564), ("coin droit", 0.7151534843120494) })
{
    double p = pentesRegion.First(r => r.nom == nomReg).pente;
    Check(Math.Abs(p - pRef) < 5e-3, $"pente régionale {nomReg} divergente du jumeau Python : {p} vs {pRef}");
}


    controle temporel : max |t_atteint - t_final| = 0,00e+00


    controle temporel : max |t_atteint - t_final| = 0,00e+00


    controle temporel : max |t_atteint - t_final| = 0,00e+00


    controle temporel : max |t_atteint - t_final| = 0,00e+00


Taux mesurés (régression log-log, N=100..1600) :


  Godunov  sur choc        : ordre 0.99


... (3 outputs left out)

Découpe régionale de l'erreur L1 (Godunov, rarefaction 0.4/1.2, memes grilles) :


  interieur du fan   : ordre 0.75


  coin gauche        : ordre 0.75


  coin droit         : ordre 0.72


**Lecture.** Les pentes imprimées ci-dessus se lisent ainsi : ordre 1 net sur le choc,
légèrement en dessous sur la raréfaction.

*Le pas de temps est borné pour finir exactement à $t_{final}$* (`dt_max` dans `avance_volume_finis`).
Sans cette borne, la boucle dépasse l'instant visé — de plusieurs pourcents sur les grilles grossières —
et l'erreur est alors comparée à une solution exacte évaluée à un **autre** instant que celui de la simulation :
l'écart de temps déplace le front, et la pente mesurée mélange l'erreur du schéma à ce désaccord d'horloge.
La ligne « contrôle temporel » imprimée par chaque étude est la preuve de l'alignement, pas une décoration.

*L'écart de pente sur la raréfaction ne s'attribue pas aux coins de l'éventail* : l'explication « ordre 1 dans le fan,
dégradation localisée aux jonctions » serait séduisante, mais elle se teste plutôt qu'elle ne s'affirme — la cellule
ci-dessus découpe la même erreur $L^1$ par région, sur la référence exacte et les mêmes grilles. Les ordres régionaux
imprimés sont du même niveau à l'intérieur du fan et aux deux coins, et **l'intérieur n'est pas la région la mieux
convergente** : cette explication est donc réfutée par les données du carnet. Ce qui reste établi, c'est le constat —
un taux global inférieur à 1 sur la raréfaction, contre l'ordre 1 net sur le choc — et le fait que l'éventail n'est pas
une donnée régulière ($C^1$) : la solution y est continue mais ses jonctions sont non lisses. Aucun mécanisme plus fin
n'est revendiqué ici ; expliquer l'écart demanderait une étude dédiée (par exemple des grilles beaucoup plus fines,
ou un schéma d'ordre 2 en exercice).
Godunov conserve une constante d'erreur plus petite que Rusanov (l'écart vertical entre les deux séries d'erreurs), sans changer
l'ordre. C'est une **mesure numérique de convergence sur cas test** — ni une preuve de convergence, ni une garantie sur
d'autres données : la Partie 4 revient précisément sur cette frontière.

## Partie 3 — Formation d'un choc : quand les caractéristiques se croisent

### 3.1 Temps de formation — dérivation

Sur une donnée régulière $u_0$, la solution classique s'écrit $u(x,t) = u_0(x_0)$ avec $x = x_0 + u_0(x_0)\,t$.
Cette paramétrisation devient singulière quand deux caractéristiques voisines se touchent :

$$\frac{\partial x}{\partial x_0} = 1 + u_0'(x_0)\,t = 0 \quad\Longrightarrow\quad t^* = -\frac{1}{\min u_0'}$$

si $\min u_0' < 0$. Pour $u_0(x) = 1 + \tfrac{1}{2}\sin(2\pi x)$, on a $u_0' = \pi\cos(2\pi x)$,
donc $\min u_0' = -\pi$ et la prédiction est $t^* = 1/\pi$.

La cellule suivante mesure $t^*$ **directement sur les caractéristiques** — aucun schéma numérique n'intervient :
on sème des particules, on regarde quand deux voisines se croisent, par dichotomie.

In [6]:
// Donnée initiale périodique sur [0, 1]
static double U0Sinus(double x) => 1.0 + 0.5 * Math.Sin(2.0 * Math.PI * x);

// Positions exactes des particules : x0 + u0(x0) * t
static double[] PositionsParticules(double t, double[] x0) => x0.Select(x0k => x0k + U0Sinus(x0k) * t).ToArray();

static double EcartMinimal(double t, double[] x0)
{
    var x = PositionsParticules(t, x0);
    return Enumerable.Range(1, x.Length - 1).Select(i => x[i] - x[i - 1]).Min();
}

// prédiction analytique : t* = -1 / min u0'(x), u0' = pi cos(2 pi x)
double stepT = 1.0 / 200000.0;   // arithmétique np.linspace(0, 1, 200001)
double tPred = -1.0 / Enumerable.Range(0, 200001).Select(k => Math.PI * Math.Cos(2.0 * Math.PI * (k * stepT))).Min();
Console.WriteLine($"prédiction analytique : t* = 1/pi = {FI(tPred)}");

// mesure par dichotomie sur le croisement de particules voisines
double stepP = 1.0 / 2000.0;   // arithmétique np.linspace(0, 1, 2001)
var x0p = Enumerable.Range(0, 2001).Select(k => k * stepP).ToArray();
double lo = 0.0, hi = 1.0;     // EcartMinimal(0) > 0, EcartMinimal(1) < 0
Check(EcartMinimal(lo, x0p) > 0.0 && EcartMinimal(hi, x0p) < 0.0, "la dichotomie exige un changement de signe");
for (int it = 0; it < 60; it++)
{
    double mid = 0.5 * (lo + hi);
    if (EcartMinimal(mid, x0p) > 0.0) lo = mid; else hi = mid;
}
double tMesure = 0.5 * (lo + hi);
double ecartRel = Math.Abs(tMesure - tPred) / tPred;
Console.WriteLine($"mesure caractéristiques : t* = {FI(tMesure)} (2001 particules, 60 dichotomies)");
Console.WriteLine($"écart relatif : {ecartRel:0.00e+00}");

// Garde exécutable : mêmes valeurs que le jumeau Python
Check(Math.Abs(tPred - 0.3183098861837907) < 1e-12, $"t_pred divergent du jumeau Python : {tPred}");
Check(Math.Abs(tMesure - 0.31831040978312963) < 1e-12, $"t_mesure divergent du jumeau Python : {tMesure}");
Check(Math.Abs(ecartRel / 1.6449358366442203e-06 - 1.0) < 1e-6, $"écart relatif divergent du jumeau Python : {ecartRel}");


prédiction analytique : t* = 1/pi = 0.318310


mesure caractéristiques : t* = 0.318310 (2001 particules, 60 dichotomies)


écart relatif : 1,64e-06


In [7]:
// la même physique, vue par le schéma de Rusanov : avant, autour, après t*
static (double[] x, double[] u) SimulePeriodique(Func<double, double> u0, double tFinal, int n = 800, double cfl = 0.9)
{
    double dx = 1.0 / n;
    var x = Enumerable.Range(0, n).Select(i => (i + 0.5) * dx).ToArray();
    double[] u = x.Select(u0).ToArray();
    double t = 0.0;
    while (t < tFinal)
    {
        double vmax = u.Max(Math.Abs);
        double dt = cfl * dx / vmax;
        if (t + dt > tFinal) dt = tFinal - t;
        double[] F = new double[n];
        for (int i = 0; i < n; i++) F[i] = FluxRusanov(u[i], u[(i + 1) % n]);   // flux aux faces DROITES
        double[] uNew = new double[n];
        for (int i = 0; i < n; i++) uNew[i] = u[i] - (dt / dx) * (F[i] - F[(i - 1 + n) % n]); // F_{i+1/2} - F_{i-1/2}
        u = uNew; t += dt;
    }
    return (x, u);
}

var panneaux = new Dictionary<double, (double[] x, double[] u)>();
foreach (double t in new[] { 0.15, tPred, 0.55 })
    panneaux[t] = SimulePeriodique(U0Sinus, t);

// verdict dérivé des trois champs calculés : raidissement vers t*, puis propagation du front
double PenteMax(double[] x, double[] u)
{
    double m = 0.0;
    for (int i = 1; i < u.Length; i++) m = Math.Max(m, Math.Abs(u[i] - u[i - 1]));
    return m / (x[1] - x[0]);
}
int IndiceFront(double[] u)
{
    // Convention np.diff : diff[i] = u[i+1] - u[i] ; argmax rend l'indice de la cellule
    // GAUCHE de l'interface la plus raide (le front est rapporté à cette cellule).
    int iMax = 0; double m = 0.0;
    for (int i = 0; i < u.Length - 1; i++)
    {
        double d = Math.Abs(u[i + 1] - u[i]);
        if (d > m) { m = d; iMax = i; }
    }
    return iMax;
}
double p015 = PenteMax(panneaux[0.15].x, panneaux[0.15].u);
double pTp  = PenteMax(panneaux[tPred].x, panneaux[tPred].u);
double facteur = pTp / p015;
double frontTp  = panneaux[tPred].x[IndiceFront(panneaux[tPred].u)];
double front055 = panneaux[0.55].x[IndiceFront(panneaux[0.55].u)];
double deplacement = Math.Abs(front055 - frontTp);
double pasGrille = panneaux[0.55].x[1] - panneaux[0.55].x[0];

Console.WriteLine($"pente max : {FI(p015, "F1")} (t=0.15) -> {FI(pTp, "F1")} (autour de t*) : "
    + (facteur > 5 ? "la solution lisse se raidit en un front raide" : "raidissement non marqué")
    + $" ; le front passe de x={FI(frontTp, "F2")} à x={FI(front055, "F2")} : "
    + (deplacement > 2 * pasGrille ? "le choc se propage" : "front stationnaire"));

// Garde exécutable : mêmes valeurs que le jumeau Python
Check(Math.Abs(p015 / 5.8935442579774575 - 1.0) < 1e-9, $"pente max t=0.15 divergente : {p015}");
Check(Math.Abs(pTp / 57.03843284490261 - 1.0) < 1e-9, $"pente max à t* divergente : {pTp}");
Check(facteur > 5.0, "le raidissement vers t* doit donner un facteur > 5 sur la pente max");
Check(Math.Abs(frontTp - 0.816875) < 1e-9, $"position du front à t* divergente : {frontTp}");
Check(Math.Abs(front055 - 0.049375) < 1e-9, $"position du front à t=0.55 divergente : {front055}");
Check(deplacement > 2 * pasGrille, "le front doit s'être déplacé de plus de deux pas de grille");


pente max : 5.9 (t=0.15) -> 57.0 (autour de t*) : la solution lisse se raidit en un front raide ; le front passe de x=0.82 à x=0.05 : le choc se propage


## Partie 4 — Ce qu'une preuve formelle établit — et ce qu'elle n'établit pas

### 4.1 Le contexte : les dépôts Lanyon et la question de #19741

L'issue #19741 pose la question d'une **preuve de correction d'un schéma numérique** face aux dépôts
d'analyses formelles dits *Lanyon*. Le document de l'étape 3 de #19741 (porté par la PR #23101, mergée le 2026-10-31 ; le document est sur `main`)
rapporte trois mesures, reprises ici **telles quelles, en qualité de mesures rapportées** :

| # | Mesure rapportée (source : PR #23101, document étape 3) | Statut |
|---|---|---|
| M1 | les fichiers de preuve des dépôts Lanyon n'importent que Mathlib, sans aucun lien avec le code C | RAPPORTÉE |
| M2 | 31 % des théorèmes de `BurgersEquation` se referment en une tactique | RAPPORTÉE |
| M3 | aucune version de Lean ni de Mathlib n'est déclarée dans ces dépôts | RAPPORTÉE |

Côté dépôt, mesure faite pour ce carnet : **aucun lake ni preuve formelle de schéma numérique n'existe sur `main`**
(recensement `git ls-tree -r origin/main` sur les motifs Burgers / volumes finis / numérique : aucun fichier de preuve).
La sous-série Numérique part donc de zéro côté formel.

### 4.2 Table des lacunes — verdict écrit par lacune (critère de sortie T3 de #23161)

| Lacune | Verdict | Raison |
|---|---|---|
| Correction du schéma discret (Godunov converge vers la solution entropique) | **NON COUVERT — grain ouvert** | les preuves Lanyon portent sur le modèle continu ; aucune ne mentionne un schéma, un pas de temps ou une grille |
| Lien preuve ↔ implémentation numérique | **NON COUVERT — grain ouvert** | M1 : ces fichiers de preuve n'ont aucun lien avec le code exécuté ; aucun pont (extraction, raffinement) déclaré |
| Sélection d'entropie dans le modèle formel | **À INVENTORIER — grain ouvert** | M2 suggère des énoncés de base ; le contenu exact (postulat d'entropie ? unicité de la solution faible ?) reste à inventorier grain par grain |
| Reproductibilité de l'environnement de preuve | **NON COUVERT — raison : hors de notre contrôle** | M3 : aucune version Lean/Mathlib déclarée dans les dépôts mesurés ; une reproduction exigerait de pinner les versions nous-mêmes |
| Reproductibilité de NOS mesures numériques | **COUVERT dans ce carnet** | schémas déterministes, aucune graine, sorties exécutées et commitées avec le carnet (règle C.2 du dépôt) |

### 4.3 Ce que ce carnet ne prouve pas

Les taux de convergence mesurés en Partie 2 sont des **mesures sur cas test** : ils corroborent la théorie
sur les données essayées, ils ne la démontrent pas. Une preuve formelle de convergence du schéma de Godunov
(théorème de Lax-Wendroff, solutions entropiques de Kruzhkov) exigerait un lake dédié dans le dépôt — c'est
exactement le grain ouvert ci-dessus. En attendant ce grain, la formulation honnête est :

> les schémas implémentés ici **mesurent** une convergence d'ordre 1 sur les cas test ;
> la preuve de correction du schéma **n'existe pas dans le dépôt** et reste un grain ouvert.

Jamais ce carnet n'écrit « vérifié de bout en bout ».

## Vérifiez votre compréhension

Les exercices suivent la règle du dépôt : le carnet reste **exécutable de bout en bout** même exercices non complétés.
Chaque squelette s'exécute sans erreur et imprime un message d'attente.

In [8]:
// Exercice 1 — Lax-Wendroff sur l'advection linéaire
//
// L'équation u_t + a u_x = 0 (a = 1) admet la solution exacte u0(x - t).
// Le schéma de Lax-Wendroff y est d'ordre 2 : l'erreur doit décroître comme dx^2.
//
// Étape 1 : implémenter un pas Lax-Wendroff périodique
//   u^{n+1}_i = u_i - (nu/2)(u_{i+1} - u_{i-1}) + (nu^2/2)(u_{i+1} - 2 u_i + u_{i-1}),
//   avec nu = a dt / dx (prendre nu = 0.8 pour rester stable).
// Étape 2 : mesurer l'erreur L1 à t = 0.3 pour N dans (100, 200, 400, 800) sur u0(x) = sin(2 pi x), périodique.
// Étape 3 : régression log-log et comparaison à la pente 2 attendue.

static double[] PasLaxWendroff(double[] u, double nu)
{
    // Un pas de Lax-Wendroff périodique. TODO étudiant
    double[] result = null; // TODO étudiant
    return result;
}

Console.WriteLine("Exercice 1 à compléter : Lax-Wendroff sur l'advection (ordre 2 attendu).");


Exercice 1 à compléter : Lax-Wendroff sur l'advection (ordre 2 attendu).


In [9]:
// Exercice 2 — Solveur de Riemann pour f(u) = u^3 / 3
//
// La loi u_t + (u^3/3)_x = 0 généralise Burgers. La condition de Rankine-Hugoniot
// (même dérivation qu'en 1.2) donne s = (uL^2 + uL uR + uR^2) / 3.
//
// Indice : pour uL = 2, uR = -1, la formule donne s = (4 - 2 + 1)/3 = 1.
//
// Étape 1 : dériver sur papier la vitesse du choc, vérifier la valeur 1 ci-dessus.
// Étape 2 : implémenter RiemannExactCubique(ul, ur, x, t) sur le modèle de BurgersRiemannExact (Partie 1).
//           ATTENTION : le flux n'est PAS convexe (f'' = 2u change de signe), donc la recette de Burgers
//           « choc si ul > ur » ne s'applique pas. Le critère porte sur les VITESSES f'(u) = u^2 :
//             - choc si f'(ul) > f'(ur), soit |ul| > |ur| (caractéristiques convergentes) ;
//               la condition d'entropie de Lax f'(ul) >= s >= f'(ur) doit être satisfaite ;
//             - éventail si f'(ul) < f'(ur), soit |ul| < |ur|.
//           Contre-exemple à la recette convexe : ul = -2, ur = -1 donne ul < ur mais |ul| > |ur| ;
//           s = (4+2+1)/3 = 7/3 et 4 >= 7/3 >= 1 : c'est un CHOC, pas un éventail.
//           Dans l'éventail, x/t = u^2 admet deux branches u = +/- sqrt(x/t) : choisir celle raccordée
//           à l'état amont (pour ul = -1, ur = 2, le point x/t = 0.5 vaut -sqrt(0.5), pas +sqrt(0.5)).
//           Le cas |ul| = |ur| est dégénéré (f'(ul) = f'(ur)) et se traite à part.
// Étape 3 : valider sur le cas (2, -1) : la discontinuité doit se trouver en x = t.

static double[] RiemannExactCubique(double ul, double ur, double[] x, double t)
{
    // Solution exacte pour f(u) = u^3/3. TODO étudiant
    double[] result = null; // TODO étudiant
    return result;
}

Console.WriteLine("Exercice 2 à compléter : solveur de Riemann pour le flux cubique.");


Exercice 2 à compléter : solveur de Riemann pour le flux cubique.


In [10]:
// Exercice 3 — Largeur de choc de Godunov
//
// La Partie 2 montre un choc étalé sur quelques cellules. Quantifions :
//
// Étape 1 : pour le Riemann (1.2, 0.4) à T = 0.5, compter pour N dans (100, 200, 400, 800, 1600)
//           le nombre de cellules où u traverse la valeur milieu (uL + uR) / 2.
// Étape 2 : vérifier que ce compte reste ~constant quand N croît, et conclure :
//           la largeur PHYSIQUE du choc décroît comme dx — c'est l'ordre 1 mesuré en Partie 2.
//
// Indice : compter les interfaces i où (u[i] - umilieu) * (u[i+1] - umilieu) < 0.

static int? CompteTraversees(double[] u, double umilieu)
{
    // Nombre d'interfaces où la solution traverse umilieu. TODO étudiant
    int? result = null; // TODO étudiant
    return result;
}

Console.WriteLine("Exercice 3 à compléter : largeur de choc de Godunov en fonction de N.");


Exercice 3 à compléter : largeur de choc de Godunov en fonction de N.


In [11]:
// Exercice 4 — Décroissance de l'entropie discrète
//
// Pour la solution entropique, l'entropie intégrée S(t) = somme_i u_i^2 dx décroît
// (production d'entropie négative au passage d'un choc). Le schéma de Godunov, monotone,
// doit hériter de cette décroissance.
//
// Étape 1 : sur le Riemann (1.2, 0.4), enregistrer S(t) à chaque pas de Godunov jusqu'à T = 0.5.
// Étape 2 : vérifier numériquement que S(t) est décroissante, et imprimer S(t).
// Étape 3 : comparer à Rusanov — même monotonie attendue.

static double[] TraceEntropie(Func<double, double, double> fluxInterface, double tFinal = 0.5, int n = 400)
{
    // Trajectoire de l'entropie discrète. TODO étudiant
    double[] result = null; // TODO étudiant
    return result;
}

Console.WriteLine("Exercice 4 à compléter : décroissance de l'entropie discrète de Godunov.");


Exercice 4 à compléter : décroissance de l'entropie discrète de Godunov.


## À retenir

| Notion | Ce qu'elle dit | Où la voir |
|---|---|---|
| Rankine–Hugoniot | $s\,(u_L-u_R) = f(u_L)-f(u_R)$ se dérive de la seule forme intégrale ; pour Burgers, $s = (u_L+u_R)/2 = 0{,}5$ sur le choc $u_L = 1$, $u_R = 0$ — vérifiée par la position du choc | Partie 1 |
| Godunov vs Rusanov | à 89 pas communs : erreur L1 de $0{,}004156$ (Godunov) contre $0{,}005131$ (Rusanov) ; ordre 1 mesuré sur le choc ($0{,}99$ pour les deux), contrôlé à $0{,}00\%$ sur le temps atteint | Partie 2 |
| Raréfaction plus dure | ordre mesuré $0{,}75$–$0{,}79$ (contre $0{,}99$ sur le choc), découpe régionale homogène : $0{,}75$ intérieur du fan, $0{,}75$ coin gauche, $0{,}72$ coin droit | Partie 2 |
| Formation d'un choc | $t^* = -1/\min u_0'$ : prédiction $1/\pi = 0{,}318310$, mesure caractéristiques $0{,}318310$ (2001 particules, 60 dichotomies), écart relatif $1{,}64 \times 10^{-6}$ | Partie 3 |
| Frontière de la preuve | les preuves formelles existantes (dépôts Lanyon, mesures rapportées M1–M3) portent sur le modèle continu, sans lien au code ; la preuve de correction d'un schéma reste un **grain ouvert** — ce carnet mesure, il ne démontre pas | Partie 4 |

La reproductibilité de nos mesures est couverte : schémas déterministes, aucune graine, sorties exécutées et commitées avec le carnet. Jamais ce carnet n'écrit « vérifié de bout en bout ».

Le jumeau C# rejoue chacune de ces valeurs avec la même précision (garde-fous exécutables dans chaque cellule).


## Pour aller plus loin

- **Carnet suivant** : [Numérique-02 — Maxwell en relativité, noyaux C et volumes finis](Numerique-02-Maxwell-Relativiste-Volumes-Finis-C.ipynb) — la même démarche de mesure appliquée aux noyaux C d'un dépôt d'analyse formelle.
- **Prolongement direct** : une fois l'exercice 1 (Lax–Wendroff) franchi, passer aux schémas d'ordre 2 (MUSCL) et mesurer ce que l'ordre ajoute près du choc — la décroissance lente mesurée sur la raréfaction ($0{,}75$) est le bon terrain de test.
- **Grains ouverts côté formel** : la table 4.2 — inventaire du contenu exact des théorèmes `BurgersEquation`, puis décision d'un lake dédié (correction de Godunov) ou d'un pont preuve/code. Suivis dans l'EPIC [#23161 — Série Physique numérique](https://github.com/jsboige/CoursIA/issues/23161).
- **Carnets voisins de la série** : [Jauge-01 — Théorème de Noether](../Jauge/Jauge-01-Theoreme-Noether-Python.ipynb), [Jauge-02 — Invariance de jauge U(1) et SU(2)](../Jauge/Jauge-02-Invariance-Jauge-U1-SU2-Python.ipynb) et [Jauge-03 — Action de Wilson et confinement](../Jauge/Jauge-03-Action-Wilson-Confinement-Python.ipynb) — l'arc Noether → Yang–Mills ; côté intégrales, la sous-série [Integrales](../Integrales/Integrales-01-Triangle-Scalaire-Python.ipynb) et son classement BootLoops.

## Références et provenance

### Provenance pédagogique

Cette sous-série s'inscrit dans l'arc de physique numérique de l'[EPIC #23161](https://github.com/jsboige/CoursIA/issues/23161), tranche T3.